::: {.sb-lesson-links}
[Exercises →](../exercises/know-your-customers.ipynb)
[Quiz →](../quiz/quiz.qmd)
[All files of this section (zip)](../../files/01-kmeans.zip)
:::

::: {.sb-downloads}
[Download this notebook](../downloads/lesson-workflow.ipynb){download="lesson-workflow.ipynb"}
:::

The practical workflow of a clustering project, on the Iris flowers: scale the data, fit K-Means, choose the number of clusters $K$ with the elbow method and the silhouette score, then give a meaning to the clusters.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.preprocessing import StandardScaler

# colour palette of the course
COLORS = ["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
pio.templates["course"] = go.layout.Template(layout_colorway=COLORS)
pio.templates.default = "plotly_white+course"

## The dataset 🌸

The [Iris dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#iris-dataset), bundled with scikit-learn, measures 150 flowers: the length and width of their sepals and petals, in cm. The flowers belong to 3 species, but we **hide the species**: we do unsupervised learning, so we only keep $X$. We will use the species at the end, only to check what K-Means found.

In [ ]:
X, species = load_iris(return_X_y=True, as_frame=True)   # we put the species aside
X.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


## Scale, then fit 🚀

K-Means uses the Euclidean distance, so we standardise the 4 variables first (mean 0, standard deviation 1 for each). Then we fit K-Means with $K=3$; we will check this choice below. As seen in the previous lesson, we keep the k-means++ start (the default) and ask for 10 runs (`n_init=10`).

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, n_init=10, random_state=0)
kmeans.fit(X_scaled)
print("cluster sizes:", np.bincount(kmeans.labels_))
print(f"inertia: {kmeans.inertia_:.1f}")

cluster sizes: [53 50 47]
inertia: 139.8


The centres live in the standardised space. To read them, we bring them back to centimetres with `inverse_transform`.

In [ ]:
centres = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=X.columns).round(2)
centres.index.name = "cluster"
fig = px.imshow(centres, text_auto=True, aspect="auto", color_continuous_scale="Blues",
                title="Centre of each cluster, in cm")
fig.update_layout(height=320)
fig.show()

**What this shows:** each row is the "typical flower" of a cluster. Cluster 1 has tiny petals (1.46 cm long, 0.25 cm wide) and the widest sepals (3.43 cm). Cluster 2 has the largest flowers (petals 5.51 cm long, sepals 6.78 cm long). Cluster 0 sits in between, with the narrowest sepals (2.67 cm). Reading the centres is the first step to give a meaning to the clusters.

A new flower is assigned to the nearest centre with `predict`. It must go through the **same** scaler first:

In [ ]:
new_flower = pd.DataFrame([[5.0, 2.0, 1.0, 0.1]], columns=X.columns)
print("the new flower belongs to cluster", kmeans.predict(scaler.transform(new_flower))[0])

the new flower belongs to cluster 0


## Choosing $K$ 🔢

K-Means needs $K$ in advance. Without a target, there is no accuracy to compute, so we use measures of the clusters themselves. There are two classic methods, which look at the clusters from two opposite angles:

* the **elbow method** checks that each cluster is **compact**: its points are close to each other;
* the **silhouette score** also checks that the clusters are **well separated**: far from each other.

### The elbow method 💪

We fit K-Means for $K=1$ to $10$ and plot the inertia of each fit.

In [ ]:
ks = range(1, 11)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_scaled).inertia_ for k in ks]
for k, inertia in zip(ks, inertias):
    gain = "" if k == 1 else f"   (-{inertias[k - 2] - inertia:.1f} compared to K={k - 1})"
    print(f"K={k:>2}: inertia {inertia:6.1f}{gain}")

fig = px.line(x=list(ks), y=inertias, markers=True, title="Elbow method: inertia for each K")
fig.update_layout(xaxis_title="number of clusters K", yaxis_title="inertia")
fig.show()

K= 1: inertia  600.0
K= 2: inertia  222.4   (-377.6 compared to K=1)
K= 3: inertia  139.8   (-82.5 compared to K=2)
K= 4: inertia  114.1   (-25.7 compared to K=3)
K= 5: inertia   90.8   (-23.3 compared to K=4)
K= 6: inertia   80.0   (-10.8 compared to K=5)
K= 7: inertia   71.6   (-8.4 compared to K=6)
K= 8: inertia   62.9   (-8.7 compared to K=7)
K= 9: inertia   55.5   (-7.3 compared to K=8)
K=10: inertia   47.3   (-8.3 compared to K=9)




::: {.callout-tip title="Intuition"}
The inertia **always** goes down when $K$ grows: with more centres, every point finds a closer one. With $K=150$ (one cluster per flower), the inertia is 0, and the clustering is useless. So we do not look for the lowest inertia, but for the **elbow**: the $K$ after which one more cluster only brings a small gain.
:::



**What this shows:** with 1 cluster, the inertia is 600 (150 flowers × 4 standardised variables). The second cluster removes 377.6 of it, the third one 82.5, the fourth one only 25.7, and the next ones even less. The curve bends at $K=3$ (some would say 4): this is the elbow. The elbow is often a matter of eye, so we check it with a second method.

### The silhouette score 👥

The silhouette asks each point: "are you closer to your own cluster than to the nearest other one?" For a point $i$ of cluster $C$:

* $a(i)$ = the mean distance between $i$ and the **other points of its own cluster** (small = cohesive cluster);
* $b(i)$ = the mean distance between $i$ and the points of the **nearest other cluster** (we compute the mean distance to each other cluster and keep the smallest);

$$
s(i)=\frac{b(i)-a(i)}{\max\big(a(i),\,b(i)\big)}
$$

$s(i)$ is always between $-1$ and $1$:

* close to **1**: the point is much closer to its own cluster than to any other, it is well placed;
* close to **0**: it is on the border between two clusters;
* **negative**: it is closer to another cluster, it is probably in the wrong one.

The **silhouette score** of a clustering is the mean of $s(i)$ over all the points. The higher, the better.



::: {.callout-tip title="Intuition"}
At a party, people form groups. For each guest, compare how far they stand from their own group ($a$) and from the nearest other group ($b$). A guest in the middle of their friends, far from the other groups, is "well placed" ($s$ near 1). A guest standing between two groups could belong to either ($s$ near 0). A guest standing next to another group is in the wrong one ($s < 0$).
:::



#### See it with data 🔬: the silhouette of one point, by hand

9 points in 3 clusters. We compute the silhouette of point $i=(2,1)$, in cluster A, step by step.

In [ ]:
toy = np.array([[1, 1], [2, 1], [1, 2],      # cluster A
                [5, 1], [6, 2], [5, 2],      # cluster B
                [3, 6], [4, 6], [3, 5]])     # cluster C
toy_labels = np.array(["A", "A", "A", "B", "B", "B", "C", "C", "C"])
i = 1                                        # the point (2, 1)

dist = np.sqrt(((toy - toy[i]) ** 2).sum(axis=1))         # Euclidean distance from i to every point
mean_dist = {c: dist[(toy_labels == c) & (np.arange(len(toy)) != i)].mean() for c in ["A", "B", "C"]}
for c, d in mean_dist.items():
    print(f"mean distance from (2, 1) to the {'other ' if c == 'A' else ''}points of cluster {c}: {d:.3f}")

a = mean_dist["A"]
b = min(mean_dist["B"], mean_dist["C"])                   # the nearest other cluster
s = (b - a) / max(a, b)
print(f"\na = {a:.3f}, b = {b:.3f}, s = (b - a) / max(a, b) = {s:.3f}")
print(f"scikit-learn silhouette_samples for this point: {silhouette_samples(toy, toy_labels)[i]:.3f}")

mean distance from (2, 1) to the other points of cluster A: 1.207
mean distance from (2, 1) to the points of cluster B: 3.428
mean distance from (2, 1) to the points of cluster C: 4.869

a = 1.207, b = 3.428, s = (b - a) / max(a, b) = 0.648
scikit-learn silhouette_samples for this point: 0.648


In [ ]:
toy_s = silhouette_samples(toy, toy_labels)
print("silhouette of each point:", toy_s.round(2))
print(f"mean of the 9 values: {toy_s.mean():.3f}   silhouette_score: {silhouette_score(toy, toy_labels):.3f}")

silhouette of each point: [0.77 0.65 0.72 0.67 0.74 0.73 0.8  0.74 0.7 ]
mean of the 9 values: 0.725   silhouette_score: 0.725


**What this shows:** the two other points of A are at distances 1 and 1.414, so $a=1.207$. Cluster B (mean distance 3.428) is nearer than cluster C (4.869), so $b=3.428$. Then $s=(3.428-1.207)/3.428=0.648$: the point is well placed. Scikit-learn's `silhouette_samples` gives the same value for this point. Its silhouette is the lowest of cluster A, because it is the point of A closest to cluster B. `silhouette_score` is simply the mean of the 9 values (0.725).

#### See it with data 🔬: what the silhouettes of a cluster tell you

Look at a cluster through two numbers: the **mean** of the silhouettes of its points, and their **spread** (standard deviation). We build four situations for the blue cluster, next to the same orange cluster: blue is either **concentrated** (all points close to its centre) or **dispersed** (a dense core plus a halo of scattered points), and either **isolated** (orange is far) or **not isolated** (orange is close).

In [ ]:
def two_clusters(dispersed, isolated):
    rng = np.random.default_rng(0)
    if dispersed:
        blue = np.vstack([rng.normal(0, 0.3, (40, 2)), rng.normal(0, 1.6, (20, 2))])   # core + halo
    else:
        blue = rng.normal(0, 0.3, (60, 2))
    orange = rng.normal([6 if isolated else 1.4, 0], 0.3, (60, 2))
    return np.vstack([blue, orange]), np.repeat([0, 1], 60)

cases = [("concentrated", "isolated"), ("concentrated", "not isolated"),
         ("dispersed", "isolated"), ("dispersed", "not isolated")]
fig = make_subplots(rows=2, cols=2, subplot_titles=[f"{a}, {b}" for a, b in cases],
                    vertical_spacing=0.12, horizontal_spacing=0.06)
rows = []
for n, (spread, isolation) in enumerate(cases):
    pts, lab = two_clusters(spread == "dispersed", isolation == "isolated")
    s_blue = silhouette_samples(pts, lab)[lab == 0]
    rows.append({"blue cluster": f"{spread}, {isolation}", "mean silhouette": s_blue.mean().round(2),
                 "spread (std)": s_blue.std().round(2), "lowest": s_blue.min().round(2)})
    fig.add_trace(go.Scatter(x=pts[:, 0], y=pts[:, 1], mode="markers", showlegend=False,
                             marker=dict(color=[COLORS[l] for l in lab], size=5)), row=n // 2 + 1, col=n % 2 + 1)
fig.update_xaxes(range=[-4.5, 7.5])
fig.update_yaxes(range=[-4.5, 4.5])
fig.update_layout(height=600, title="Four shapes for the blue cluster")
fig.show()
pd.DataFrame(rows)

,blue cluster,mean silhouette,spread (std),lowest
0,"concentrated, isolated",0.91,0.02,0.85
1,"concentrated, not isolated",0.61,0.14,0.15
2,"dispersed, isolated",0.76,0.12,0.28
3,"dispersed, not isolated",0.17,0.23,-0.45


**What this shows:**

| | isolated | not isolated |
|---|---|---|
| **concentrated** | high mean (0.91), tiny spread (0.02) | lower mean (0.61), small spread (0.14) |
| **dispersed** | high mean (0.76), larger spread (0.12) | low mean (0.17), large spread (0.23) |

* **Isolation drives the mean:** when orange comes close, $b(i)$ shrinks for every blue point, and the mean silhouette drops (0.91 → 0.61, and 0.76 → 0.17).
* **Dispersion drives the spread:** the halo points are far from their own centre ($a(i)$ is large for them, small for the core), so the silhouettes become uneven: the spread grows (0.02 → 0.12 when isolated, 0.14 → 0.23 when not). In the worst case, some halo points even get a negative silhouette.

A good cluster has a high mean **and** a small spread: it is concentrated and isolated.

### Silhouette on the Iris flowers

Back to the flowers: the silhouette score for $K=2$ to $10$ (it needs at least 2 clusters).

In [ ]:
ks = range(2, 11)
sil = []
for k in ks:
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X_scaled)
    sil.append(silhouette_score(X_scaled, labels))
    print(f"K={k:>2}: silhouette score {sil[-1]:.3f}")

fig = px.bar(x=list(ks), y=sil, title="Silhouette score for each K")
fig.update_layout(xaxis_title="number of clusters K", yaxis_title="silhouette score")
fig.show()

K= 2: silhouette score 0.582
K= 3: silhouette score 0.460
K= 4: silhouette score 0.387


K= 5: silhouette score 0.346
K= 6: silhouette score 0.326
K= 7: silhouette score 0.332
K= 8: silhouette score 0.345
K= 9: silhouette score 0.340


K=10: silhouette score 0.349


**What this shows:** the silhouette score is highest for $K=2$ (0.582), then $K=3$ (0.460), and it drops for larger $K$. The elbow suggested 3. The two methods disagree! A silhouette plot helps to understand why: for each cluster, we draw the silhouette of each point, sorted, as a horizontal bar.

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=["K = 2", "K = 3"], shared_yaxes=True)
for col, k in [(1, 2), (2, 3)]:
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X_scaled)
    s_all = silhouette_samples(X_scaled, labels)
    y0 = 0
    for c in range(k):
        values = np.sort(s_all[labels == c])
        fig.add_trace(go.Bar(x=values, y=np.arange(y0, y0 + len(values)), orientation="h", width=1,
                             marker_color=COLORS[c], showlegend=False), row=1, col=col)
        print(f"K={k}, cluster {c}: {len(values):>3} flowers, mean silhouette {values.mean():.2f}, "
              f"negative values: {(values < 0).sum()}")
        y0 += len(values) + 8
    fig.add_vline(x=s_all.mean(), line_dash="dash", row=1, col=col)
fig.update_xaxes(title_text="silhouette of each flower (dashed line: silhouette score)")
fig.update_yaxes(showticklabels=False)
fig.update_layout(height=450, title="Silhouette plots", bargap=0)
fig.show()

K=2, cluster 0:  50 flowers, mean silhouette 0.68, negative values: 0
K=2, cluster 1: 100 flowers, mean silhouette 0.53, negative values: 0
K=3, cluster 0:  53 flowers, mean silhouette 0.39, negative values: 0
K=3, cluster 1:  50 flowers, mean silhouette 0.64, negative values: 0
K=3, cluster 2:  47 flowers, mean silhouette 0.35, negative values: 2


**What this shows:** with $K=2$, one cluster of 50 flowers is very well separated (mean silhouette 0.68), and the other one, with 100 flowers, is fine too (0.53). With $K=3$, the same 50 flowers keep their cluster (0.64), but the 100 others are cut in two clusters whose silhouettes are much lower (0.39 and 0.35), with 2 negative values: these two clusters touch each other. So the data contains one very clear group, plus a large group that **can** be split in two, but without a clean gap.

Here we know the truth: let's compare the clusters with the species we put aside.

In [ ]:
names = load_iris().target_names
for k in [2, 3]:
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X_scaled)
    print(f"--- K = {k} ---")
    print(pd.crosstab(pd.Series(labels, name="cluster"), pd.Series(names[species], name="species")), "\n")

--- K = 2 ---
species  setosa  versicolor  virginica
cluster                               
0            50           0          0
1             0          50         50 

--- K = 3 ---
species  setosa  versicolor  virginica
cluster                               
0             0          39         14
1            50           0          0
2             0          11         36 



**What this shows:** the very clear group is the *setosa* species: all 50 flowers, alone in their cluster. *Versicolor* and *virginica* look alike, so with $K=2$ they share a cluster. With $K=3$, K-Means separates them fairly well, but not perfectly: cluster 0 holds 39 *versicolor* and 14 *virginica*, cluster 2 holds 36 *virginica* and 11 *versicolor*. 25 flowers out of 100 end up with the other species.

So which $K$ is right? Both answers say something true about the data. The elbow and the silhouette are **guides, not judges**: use both, then choose with your knowledge of the field and of what the clusters will be used for. Here, a botanist would take $K=3$.

## Interpreting the clusters 🔎

A clustering is only useful if you can **say what each cluster is**, and link it to a concrete reality ("small flowers with wide sepals", "young customers who spend a lot"...). Three tools:

1. **Read the centres**, in the original units, as we did above.
2. **Check that the centres are good representatives** of their cluster: if a cluster is very spread out, its centre hides a lot of variety.
3. **Look at the clusters**: directly when there are few variables, or after reducing them to 2 dimensions when there are many.

In [ ]:
clustered = X.assign(cluster=kmeans.labels_.astype(str))
long = clustered.melt(id_vars="cluster", var_name="variable", value_name="cm")
fig = px.box(long, x="variable", y="cm", color="cluster", category_orders={"cluster": ["0", "1", "2"]},
             title="Distribution of each variable inside each cluster (K = 3)")
fig.show()

**What this shows:** for the petals, the three clusters are clearly ordered (cluster 1 tiny, cluster 0 medium, cluster 2 large), and the boxes are narrow compared with the gaps between them: the centres are faithful summaries, and the petal size is what separates the clusters most. The sepals add a nuance: cluster 0 has the narrowest sepals and cluster 1 the widest. Their whiskers overlap more, though: the sepals alone would not separate the clusters as well.

With 4 variables, we can still look at all the pairs of variables at once:

In [ ]:
fig = px.scatter_matrix(clustered, dimensions=X.columns, color="cluster",
                        category_orders={"cluster": ["0", "1", "2"]}, title="The 3 clusters, two variables at a time")
fig.update_traces(diagonal_visible=False, marker=dict(size=4))
fig.update_layout(height=650)
fig.show()

With 20 variables, there would be 190 pairs: too many to look at. The usual solution is to project the data on the 2 directions that keep most of its variety, with a **Principal Component Analysis (PCA)**. PCA is the subject of section 3; here is just a preview:

In [ ]:
pca = PCA(n_components=2).fit(X_scaled)
coords = pca.transform(X_scaled)
print(f"share of the variance kept by the 2 components: {pca.explained_variance_ratio_.sum():.1%}")
fig = px.scatter(x=coords[:, 0], y=coords[:, 1], color=clustered.cluster, category_orders={"color": ["0", "1", "2"]},
                 labels={"x": "component 1", "y": "component 2", "color": "cluster"},
                 title="The clusters seen in the plane of the first two principal components")
fig.show()

share of the variance kept by the 2 components: 95.8%


**What this shows:** one picture summarises 4 variables while keeping 95.8% of their variance. Cluster 1 is clearly apart; clusters 0 and 2 touch each other, as the silhouette plot told us.

Keep in mind that K-Means gets harder to interpret as the number of variables grows: the distances between points grow and become similar, and there are more and more centre coordinates to read. Choose the variables you cluster on with care.

## Summary ✅

1. **Scale** the variables (`StandardScaler`), then fit `KMeans(n_clusters=K, n_init=10, random_state=0)`.
2. **Choose $K$** with the elbow method (inertia, look for the bend) **and** the silhouette score (the higher the better), then decide with your knowledge of the field.
3. **Interpret** the clusters: read the centres in the original units (`inverse_transform`), check the spread inside each cluster, and visualise them (directly, or with PCA when there are many variables).

## Resources 📚

* [`KMeans`, scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
* [Selecting the number of clusters with silhouette analysis, scikit-learn example](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html)
* [Silhouettes: P. J. Rousseeuw, *Journal of Computational and Applied Mathematics*, 1987](https://doi.org/10.1016/0377-0427(87)90125-7)

::: {.sb-next}
**[Your turn: K-Means exercises →](../exercises/know-your-customers.ipynb)**
:::